In [ ]:
# 8. Save Export Artifacts - Save the files used by the Hugging Face upload step.
np.save('response_embeddings.npy', prompt_embeddings)

with open('response_texts.pkl', 'wb') as f:
    pickle.dump(pairs_df['assistant_reply'].tolist(), f)

with open('emotion_labels.pkl', 'wb') as f:
    pickle.dump(pairs_df['emotion'].tolist(), f)

with open('response_bank_clean_export.json', 'w', encoding='utf-8') as f:
    json.dump(bank_df.to_dict(orient='records'), f, ensure_ascii=False, indent=2)

with open('retrieval-based-response-system.txt', 'w') as f:
    f.write(MODEL_NAME)

print('Saved: response_embeddings.npy')
print('Saved: response_texts.pkl')
print('Saved: emotion_labels.pkl')
print('Saved: response_bank_clean_export.json')
print('Saved: retrieval-based-response-system.txt')

# 9. Push to Hugging Face - Upload saved retrieval artifacts and the model reference file.
# Run this only after you save the files you want to publish.
from huggingface_hub import HfApi, login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret('HF_TOKEN')

login(token=secret_value_0)

api = HfApi()
repo_id = 'melisaolivia18/empathetic-retrieval-sbert'

api.create_repo(repo_id=repo_id, exist_ok=True)

api.upload_file(
    path_or_fileobj='response_embeddings.npy',
    path_in_repo='response_embeddings.npy',
    repo_id=repo_id,
)
api.upload_file(
    path_or_fileobj='response_texts.pkl',
    path_in_repo='response_texts.pkl',
    repo_id=repo_id,
)
api.upload_file(
    path_or_fileobj='emotion_labels.pkl',
    path_in_repo='emotion_labels.pkl',
    repo_id=repo_id,
)
api.upload_file(
    path_or_fileobj='retrieval-based-response-system.txt',
    path_in_repo='retrieval-based-response-system.txt',
    repo_id=repo_id,
)

print(f'\nAll files pushed to https://huggingface.co/{repo_id}')

In [1]:
# ==============================
# 0. Imports
# Core libraries for loading cleaned data, building embeddings, and retrieval.
# ==============================
import json
import pickle
import random

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# ==============================
# 1. Configuration
# Use only the two cleaned artifacts from the preprocessing pipeline.
# ==============================
TRAINING_PAIRS_PATH = '/kaggle/input/datasets/melisaolivia/nlp-music-music-cleaned/training_pairs_clean.csv'
RESPONSE_BANK_PATH = '/kaggle/input/datasets/melisaolivia/nlp-music-music-cleaned/response_bank_clean.json'
MODEL_NAME = 'all-MiniLM-L6-v2'
DEFAULT_THRESHOLD = 0.45
random.seed(42)

print('Training pairs :', TRAINING_PAIRS_PATH)
print('Response bank  :', RESPONSE_BANK_PATH)
print('Model          :', MODEL_NAME)

Training pairs : /kaggle/input/datasets/melisaolivia/nlp-music-music-cleaned/training_pairs_clean.csv
Response bank  : /kaggle/input/datasets/melisaolivia/nlp-music-music-cleaned/response_bank_clean.json
Model          : all-MiniLM-L6-v2


In [3]:
# ==============================
# 2. Helper Functions
# Normalize emotion labels so the classifier output matches the dataset labels.
# ==============================
def normalize_emotion(emotion):
    if emotion is None:
        return ''
    emotion = str(emotion).strip().lower()
    aliases = {
        'sadness': 'sad',
        'anger': 'angry',
        'joy': 'joyful',
        'anxiety': 'anxious',
        'calm': 'content',
    }
    return aliases.get(emotion, emotion)


def allowed_emotion_labels(emotion):
    emotion = normalize_emotion(emotion)
    mapping = {
        'sad': ['sad', 'lonely', 'guilty', 'disappointed', 'devastated'],
        'angry': ['angry', 'furious', 'annoyed', 'disgusted'],
        'joyful': ['joyful', 'excited', 'proud', 'grateful', 'hopeful'],
        'anxious': ['anxious', 'afraid', 'terrified', 'apprehensive'],
        'content': ['content', 'confident', 'prepared', 'sentimental'],
    }
    return mapping.get(emotion, [emotion])


def standardize_pairs_frame(df):
    required = ['user_message', 'assistant_reply', 'emotion']
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f'Missing required columns: {missing}. Found: {list(df.columns)}')

    df = df[required].dropna().copy()
    df['user_message'] = df['user_message'].astype(str).str.strip()
    df['assistant_reply'] = df['assistant_reply'].astype(str).str.strip()
    df['emotion'] = df['emotion'].map(normalize_emotion)
    df = df[(df['user_message'] != '') & (df['assistant_reply'] != '') & (df['emotion'] != '')]
    return df.drop_duplicates().reset_index(drop=True)


def load_response_bank(path):
    with open(path, 'r', encoding='utf-8') as f:
        bank = json.load(f)

    rows = []
    for emotion, replies in bank.items():
        emotion_key = normalize_emotion(emotion)
        for reply in replies:
            reply = str(reply).strip()
            if reply:
                rows.append({'emotion': emotion_key, 'assistant_reply': reply})

    bank_df = pd.DataFrame(rows).drop_duplicates().reset_index(drop=True)
    if bank_df.empty:
        raise ValueError(f'No responses found in {path}')
    return bank_df

In [4]:
# ==============================
# 3. Load Cleaned Data
# Read the cleaned training pairs and cleaned response bank.
# ==============================
pairs_df = standardize_pairs_frame(pd.read_csv(TRAINING_PAIRS_PATH))
bank_df = load_response_bank(RESPONSE_BANK_PATH)

print('Training pairs rows:', len(pairs_df))
print(pairs_df.head())

print('\nResponse bank rows:', len(bank_df))
print(bank_df.head())

print('\nTraining pair emotions:')
print(pairs_df['emotion'].value_counts().head(15))

Training pairs rows: 16661
                                        user_message  \
0  I remember going to see the fireworks with my ...   
1                This was a best friend. I miss her.   
2                                 We no longer talk.   
3  it feels like hitting to blank wall when i see...   
4   Yea it hasn't been easy but I am proud I haven't   

                                     assistant_reply      emotion  
0  Was this a friend you were in love with, or ju...  sentimental  
1                                Where has she gone?  sentimental  
2  Oh was this something that happened because of...  sentimental  
3                      Oh ya? I don't really see how       afraid  
4  What do you mean it hasn't been easy? How clos...     faithful  

Response bank rows: 4809
  emotion                                    assistant_reply
0       (  Sorry to hear that. You must be going through ...
1  afraid  Yikes! Did you get home okay? Normally I weave...
2  afraid       Tha

In [5]:
# ==============================
# 4. Build Embedding Retriever
# Encode user messages from the cleaned training pairs for similarity search.
# ==============================
sbert_model = SentenceTransformer(MODEL_NAME)

print('Encoding user_message embeddings...')
prompt_embeddings = sbert_model.encode(
    pairs_df['user_message'].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
)

print('Embedding matrix shape:', prompt_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding user_message embeddings...


Batches:   0%|          | 0/261 [00:00<?, ?it/s]

Embedding matrix shape: (16661, 384)


In [6]:
# ==============================
# 5. Retrieval Logic
# Retrieve one response from the cleaned pairs, then fall back to the
# cleaned response bank if similarity is weak or no label bucket exists.
# ==============================
DEFAULT_RESPONSES = {
    'sad': "I'm really sorry you're going through that. I'm here with you.",
    'angry': "That sounds really frustrating. I'm here to listen.",
    'joyful': "I'm glad to hear that. That sounds really meaningful.",
    'anxious': "That sounds overwhelming. Let's take it one step at a time.",
    'content': "I'm glad you're feeling at peace right now.",
}


def fallback_response(detected_emotion):
    emotion_key = normalize_emotion(detected_emotion)
    allowed = allowed_emotion_labels(emotion_key)
    bank_rows = bank_df[bank_df['emotion'].isin(allowed)]
    if not bank_rows.empty:
        row = bank_rows.sample(1, random_state=random.randint(0, 10000)).iloc[0]
        return row['assistant_reply']
    return DEFAULT_RESPONSES.get(emotion_key, "That sounds important. I'm here with you.")


def retrieve_response(user_input, detected_emotion=None, threshold=DEFAULT_THRESHOLD):
    emotion_key = normalize_emotion(detected_emotion)
    allowed = allowed_emotion_labels(emotion_key)

    user_embedding = sbert_model.encode([user_input], convert_to_numpy=True)
    scores = cosine_similarity(user_embedding, prompt_embeddings)[0]

    candidate_indices = pairs_df.index[pairs_df['emotion'].isin(allowed)].tolist()
    if not candidate_indices:
        return fallback_response(emotion_key), 0.0, 'fallback_bank'

    candidate_scores = scores[candidate_indices]
    best_pos = int(np.argmax(candidate_scores))
    best_idx = candidate_indices[best_pos]
    best_score = float(candidate_scores[best_pos])

    if best_score < threshold:
        return fallback_response(emotion_key), best_score, 'fallback_bank'

    matched_emotion = pairs_df.iloc[best_idx]['emotion']
    response = pairs_df.iloc[best_idx]['assistant_reply']
    return response, best_score, matched_emotion

In [7]:
# ==============================
# 6. Response Composer
# Turn the retrieved empathetic reply into the final chatbot answer
# by adding one supportive sentence and a music recommendation line.
# ==============================
def compose_final_response(emotion, retrieved_response):
    emotion = str(emotion).strip().lower()

    emotion_templates = {
        'sadness': {
            'follow_up': 'I hope things get a little easier for you soon.',
            'music': 'I have a music recommendation that may comfort you.'
        },
        'anger': {
            'follow_up': 'Your feelings are valid, and it is okay to feel upset.',
            'music': 'I have a music recommendation that matches your mood.'
        },
        'joy': {
            'follow_up': 'I hope this happy moment brings you even more beautiful experiences ahead.',
            'music': 'I have a music recommendation that matches your lovely mood!'
        },
        'anxiety': {
            'follow_up': 'I hope you can take things one step at a time and be gentle with yourself.',
            'music': 'I have a music recommendation that may help you feel a little calmer.'
        },
        'calm': {
            'follow_up': 'It is really nice to hear that you are feeling peaceful right now.',
            'music': 'I have a music recommendation that matches your calm mood.'
        }
    }

    fallback_template = {
        'follow_up': 'Thank you for sharing how you feel.',
        'music': 'I have a music recommendation based on your mood.'
    }

    template = emotion_templates.get(emotion, fallback_template)

    retrieved_response = str(retrieved_response).strip()
    if not retrieved_response:
        retrieved_response = 'Thank you for sharing how you feel.'

    if not retrieved_response.endswith(('.', '!', '?')):
        retrieved_response += '.'

    return f"{retrieved_response} {template['follow_up']} {template['music']}"


def chatbot_reply(user_input, detected_emotion):
    retrieved_response, score, matched_via = retrieve_response(
        user_input,
        detected_emotion=detected_emotion
    )

    final_response = compose_final_response(
        emotion=detected_emotion,
        retrieved_response=retrieved_response
    )

    return {
        'input': user_input,
        'emotion': detected_emotion,
        'matched_via': matched_via,
        'score': score,
        'retrieved_response': retrieved_response,
        'final_response': final_response
    }

In [8]:
# ==============================
# 7. Run Main Test Cases
# Show both the retrieved base response and the final chatbot-style response.
# ==============================
test_cases = [
    ("i'm feeling down, missing my ex-crush. lately, there have been so many things that remind me of him.", 'sadness'),
    ('I am feel angry when my ex situasionship keep talking about me', 'anger'),
    ('i feel happy because i finally found someone who treats me right in relationship', 'joy'),
    ('I feel anxious about my future, like economy, world war, and bad government.', 'anxiety'),
    ('I feel calm and at peace with everything.', 'calm'),
]

print('\n=== Retrieval Test ===')
for text, emotion in test_cases:
    result = chatbot_reply(text, emotion)
    print(f'\nInput              : {result['input']}')
    print(f'Emotion            : {result['emotion']} -> matched via [{result['matched_via']}]')
    print(f'Score              : {result['score']:.4f}')
    print(f'Retrieved response : {result['retrieved_response']}')
    print(f'Final response     : {result['final_response']}')


=== Retrieval Test ===

Input              : i'm feeling down, missing my ex-crush. lately, there have been so many things that remind me of him.
Emotion            : sadness -> matched via [lonely]
Score              : 0.6890
Retrieved response : thats no good, are you going to try to contact them again?
Final response     : thats no good, are you going to try to contact them again? I hope things get a little easier for you soon. I have a music recommendation that may comfort you.

Input              : I am feel angry when my ex situasionship keep talking about me
Emotion            : anger -> matched via [annoyed]
Score              : 0.5946
Retrieved response : What did she or he do lately?
Final response     : What did she or he do lately? Your feelings are valid, and it is okay to feel upset. I have a music recommendation that matches your mood.

Input              : i feel happy because i finally found someone who treats me right in relationship
Emotion            : joy -> match

In [9]:
# ==============================
# 8. Save Export Artifacts
# Save the files used by the Hugging Face upload step.
# ==============================
np.save('response_embeddings.npy', prompt_embeddings)

with open('response_texts.pkl', 'wb') as f:
    pickle.dump(pairs_df['assistant_reply'].tolist(), f)

with open('emotion_labels.pkl', 'wb') as f:
    pickle.dump(pairs_df['emotion'].tolist(), f)

with open('response_bank_clean_export.json', 'w', encoding='utf-8') as f:
    json.dump(bank_df.to_dict(orient='records'), f, ensure_ascii=False, indent=2)

with open('retrieval-based-response-system.txt', 'w') as f:
    f.write(MODEL_NAME)

print('Saved: response_embeddings.npy')
print('Saved: response_texts.pkl')
print('Saved: emotion_labels.pkl')
print('Saved: response_bank_clean_export.json')
print('Saved: retrieval-based-response-system.txt')

Saved: response_embeddings.npy
Saved: response_texts.pkl
Saved: emotion_labels.pkl
Saved: response_bank_clean_export.json
Saved: retrieval-based-response-system.txt


In [10]:
# ==============================
# 9. Push to Hugging Face
# Upload saved retrieval artifacts and the model reference file.
# Run this only after you save the files you want to publish.
# ==============================
from huggingface_hub import HfApi, login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret('HF_TOKEN')

login(token=secret_value_0)

api = HfApi()
repo_id = 'melisaolivia18/empathetic-retrieval-sbert'

api.create_repo(repo_id=repo_id, exist_ok=True)

api.upload_file(
    path_or_fileobj='response_embeddings.npy',
    path_in_repo='response_embeddings.npy',
    repo_id=repo_id,
)
api.upload_file(
    path_or_fileobj='response_texts.pkl',
    path_in_repo='response_texts.pkl',
    repo_id=repo_id,
)
api.upload_file(
    path_or_fileobj='emotion_labels.pkl',
    path_in_repo='emotion_labels.pkl',
    repo_id=repo_id,
)
api.upload_file(
    path_or_fileobj='retrieval-based-response-system.txt',
    path_in_repo='retrieval-based-response-system.txt',
    repo_id=repo_id,
)

print(f'\nAll files pushed to https://huggingface.co/{repo_id}')

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.



All files pushed to https://huggingface.co/melisaolivia18/empathetic-retrieval-sbert
